# Corsia on a Colab T4

Three runs of Qwen3-30B-A3B (a mixture-of-experts model) on a T4 GPU:

1. **Fits on the GPU**: the 2-bit model (11.8 GB) entirely on the T4. Plain llama.cpp, no Corsia. This is the fast baseline.
2. **Too big, stock**: the 4-bit model (about 18.6 GB) is bigger than the T4's memory and Colab's RAM. Attention runs on the GPU, the experts stay on the CPU, memory-mapped from disk.
3. **Too big, Corsia**: the same, with Corsia streaming the experts into its expert pool (run twice: cold, then warm from the usage profile the first run saves).

Before starting: **Runtime > Change runtime type > T4 GPU**, then run the cells in order. Cell 2 asks you to upload `corsia-colab.tar.gz`. Building takes about 15 to 20 minutes, the downloads about 30 GB.

In [ ]:
# 1. What this machine has
!nvidia-smi --query-gpu=name,memory.total --format=csv
!free -g | head -2
!nproc
!lscpu | grep -m1 'Model name'
!df -h /content | tail -1
!ls -l /dev/fuse || echo 'No /dev/fuse: Corsia cannot mount here (run 3 will not work)'

In [ ]:
# 2. Upload the Corsia bundle (corsia-colab.tar.gz), then unpack it
import os
if not os.path.exists('/content/corsia-colab.tar.gz'):
    from google.colab import files
    files.upload()
!tar xzf /content/corsia-colab.tar.gz -C /content && ls /content/corsia

In [ ]:
# 3. Tools: FUSE for Corsia's mount, Rust to build Corsia
!apt-get -qq update > /dev/null && apt-get -qq install -y fuse3 > /dev/null && echo 'fuse3 installed'
!curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal > /dev/null 2>&1 && echo 'rust installed'
import os
os.environ['PATH'] = '/root/.cargo/bin:' + os.environ['PATH']
!cargo --version

In [ ]:
# 4. Build Corsia
!cd /content/corsia && cargo build --release -q 2>&1 | tail -3
!ls -la /content/corsia/target/release/corsia

In [ ]:
# 5. Build llama.cpp b12283 with the Corsia patches, for the T4 (CUDA, compute 7.5). About 15 minutes.
!test -d /content/llama.cpp || git clone -q --depth 1 --branch b12283 https://github.com/ggml-org/llama.cpp.git /content/llama.cpp
!cd /content/llama.cpp && for p in /content/corsia/integrations/llama.cpp/000*.patch; do git apply --check "$p" 2>/dev/null && git apply "$p" && echo "applied $(basename $p)"; done
!cd /content/llama.cpp && cmake -B build -DCMAKE_BUILD_TYPE=Release -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 -DLLAMA_CURL=OFF -DLLAMA_BUILD_TESTS=OFF > /dev/null && cmake --build build --target llama-completion -j $(( $(nproc) * 2 )) 2>&1 | tail -2
!ls -la /content/llama.cpp/build/bin/llama-completion

In [ ]:
# 6. Download the two models (about 30 GB)
!pip -q install -U huggingface_hub > /dev/null
from huggingface_hub import hf_hub_download, list_repo_files
REPO = 'unsloth/Qwen3-30B-A3B-GGUF'
names = [f for f in list_repo_files(REPO) if f.endswith('.gguf')]
print('In the repository:', ', '.join(sorted(names)))
SMALL = 'Qwen3-30B-A3B-UD-Q2_K_XL.gguf'   # fits on the T4
BIG = 'Qwen3-30B-A3B-Q4_K_M.gguf'         # does not fit: pick another Q4 name from the list above if this one is missing
small_path = hf_hub_download(REPO, SMALL, local_dir='/content/models')
big_path = hf_hub_download(REPO, BIG, local_dir='/content/models')
!ls -la /content/models/*.gguf

In [ ]:
# 7. Helpers: run llama.cpp and read its timings; start and stop Corsia
import os, re, signal, subprocess, time

LLAMA = '/content/llama.cpp/build/bin/llama-completion'
CORSIA = '/content/corsia/target/release/corsia'
PROMPT = 'Write a short story about a lighthouse keeper.'
results = []

def cold(path):
    # Drop the file from the page cache so every run starts the same way.
    fd = os.open(path, os.O_RDONLY)
    os.posix_fadvise(fd, 0, 0, os.POSIX_FADV_DONTNEED)
    os.close(fd)

def run_llama(label, model, extra, env_extra=None, tokens=64):
    env = dict(os.environ)
    env.update(env_extra or {})
    cmd = [LLAMA, '-m', model, '-p', PROMPT, '-n', str(tokens), '-no-cnv', '--no-warmup', '-c', '4096', '-fa', 'on'] + extra
    t0 = time.time()
    p = subprocess.run(cmd, env=env, capture_output=True, text=True, timeout=7200)
    out = p.stdout + p.stderr
    ev = re.findall(r'(?<!prompt )eval time =\s*[\d.]+ ms /\s*\d+ runs\s*\(\s*([\d.]+) ms per token,\s*([\d.]+) tokens per second', out)
    pp = re.findall(r'prompt eval time =\s*([\d.]+) ms /\s*(\d+) tokens', out)
    r = {'run': label,
         'generate tok/s': float(ev[-1][1]) if ev else None,
         'prompt s': round(float(pp[-1][0]) / 1000, 1) if pp else None,
         'wall s': round(time.time() - t0)}
    results.append(r)
    print(r)
    for line in out.splitlines():
        if 'corsia:' in line or 'error' in line.lower() or 'failed' in line.lower():
            print('  ', line.strip()[-200:])
    if not ev:
        print(out[-3000:])
    return r

def start_corsia(model):
    mnt = '/content/corsia-mnt'
    subprocess.run(['fusermount3', '-u', mnt], capture_output=True)
    os.makedirs(mnt, exist_ok=True)
    log_path = '/content/corsia-serve.log'
    log = open(log_path, 'w')
    p = subprocess.Popen([CORSIA, 'serve', model, '--mount', mnt, '--port', '7714', '--cache-mib', '16'],
                         stdout=log, stderr=subprocess.STDOUT)
    for _ in range(240):
        time.sleep(0.5)
        text = open(log_path).read()
        if 'Press Ctrl+C' in text or p.poll() is not None:
            break
    text = open(log_path).read()
    m = re.search(r'Engines that speak Corsia: (.*?)  \(hinted', text)
    if p.poll() is not None or not m:
        print(text)
        raise RuntimeError('Corsia did not start (see its log above)')
    print('Corsia is serving', os.path.basename(model), 'on', m.group(1))
    return p, m.group(1)

def stop_corsia(p):
    p.send_signal(signal.SIGINT)
    try:
        p.wait(60)
    except subprocess.TimeoutExpired:
        p.kill()

## Run 1: the 2-bit model, all on the GPU

Plain llama.cpp: no Corsia. This is what a model that fits on the GPU does.

In [ ]:
cold(small_path)
run_llama('1 fits on GPU (Q2, all layers on the T4)', small_path, ['-ngl', '99'])

## Run 2: the 4-bit model, too big, stock

Attention on the GPU (`-ngl 99`), experts on the CPU (`--cpu-moe`), memory-mapped from disk. Run twice: cold, then with whatever the page cache kept.

In [ ]:
cold(big_path)
run_llama('2a too big, stock, cold', big_path, ['-ngl', '99', '--cpu-moe'])
run_llama('2b too big, stock, second run', big_path, ['-ngl', '99', '--cpu-moe'])

## Run 3: the 4-bit model, too big, with Corsia

The same, with Corsia's expert pool: experts are read by Corsia straight into memory llama.cpp uses, the most used stay, and swapping prefers experts already in memory (it changes the output very slightly; set `CORSIA_SWAP` to `'0'` below for output identical to stock). The first run saves a usage profile, the second starts warm from it.

`POOL_MIB` is the pool's size: about half of this machine's RAM is a good start. Lower it if the run is killed for memory.

In [ ]:
POOL_MIB = '6000'
corsia, sock = start_corsia(big_path)
try:
    env = {'CORSIA_SOCKET': sock, 'CORSIA_POOL_MIB': POOL_MIB}
    cold(big_path)
    run_llama('3a too big, Corsia pool, cold', big_path, ['-ngl', '99', '--cpu-moe'], env)
    run_llama('3b too big, Corsia pool, warm', big_path, ['-ngl', '99', '--cpu-moe'], env)
finally:
    stop_corsia(corsia)

In [ ]:
# 8. Summary
import pandas as pd
pd.DataFrame(results).set_index('run')

### Reading the results

- Run 1 against runs 2 and 3: what it costs when a model does not fit on the GPU.
- Run 2 against run 3: what Corsia adds when it does not fit. Look at `generate tok/s` and `prompt s`.
- The `corsia:` lines under each Corsia run show how many experts were found in the pool and how much was read.
- Please send back the summary table and the `corsia:` lines.